# Supply-side and unlock-event analysis (tokenomist)

**Companion to `01_pilot_walkthrough.ipynb`.** Notebook 1 covers the price-side analysis (Findings 1-3 in the README, fig1 and fig2). This notebook covers the supply-side analysis using tokenomist's API.

What this notebook does:

1. Pulls the full token list from tokenomist via the API. Free trial returns 50 tokens; paid tiers return all ~410.
2. Computes a supply-side breakdown: locked vs. unlocked vs. untracked share for each covered token.
3. Intersects with the CoinGecko universe used in notebook 1.
4. Pulls historical unlock events for the intersection.
5. Produces three supplementary figures: (3) the unlock-event calendar, (4) cross-sectional locked-share distribution, (5) cumulative dilution over the lookback window — the supply-side parallel to the price-side mismeasurement findings in notebook 1.

**Sample size note.** With the free-trial 50-token cap and the resulting small intersection with the CoinGecko universe, the sample here is too small for predictive tests. The figures are descriptive and intended to (i) show that the supply-side data infrastructure is in place and (ii) demonstrate the magnitude of supply pressure individual tokens have absorbed. The full project will use a paid Standard tier for full coverage.

## Setup

In [ ]:
import os
from pathlib import Path

env_path = Path('..') / '.env'
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        k, v = line.split('=', 1)
        os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

key = os.environ.get('TOKENOMIST_API_KEY', '').strip()
if not key:
    raise RuntimeError('TOKENOMIST_API_KEY not found. Add it to .env and re-run.')
print(f'tokenomist key loaded: {key[:5]}...{key[-3:]}')

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from IPython.display import Image

from src import tokenomist, plots

# We need the CoinGecko symbol universe to compute the intersection. Load from
# the panel.csv that notebook 1 produces. Run notebook 1 first if this file
# does not exist.
panel = pd.read_csv('../data/panel.csv')
cg_symbols = {s.upper() for s in panel['symbol'].dropna()}
print(f'CoinGecko universe (from notebook 1): {len(cg_symbols)} tokens')

## 1. Token list and tier detection

In [ ]:
tokens_df = tokenomist.list_tokens()
print(f'tokenomist coverage: {len(tokens_df)} tokens ({tokenomist.detect_tier(tokens_df)})')
tokens_df.head()

## 2. Supply-side breakdown

Locked / unlocked / untracked amounts per token from tokenomist's own labels. The `locked_share_of_trackable` column is the supply-side overhang measure summarized in `fig4_locked_share.png` below.

In [ ]:
supply_df = tokenomist.supply_breakdown(tokens_df)
supply_df.to_csv('../data/tokenomist_supply.csv', index=False)
supply_df.sort_values('locked_share_of_trackable', ascending=False).head(15)

## 3. Intersection with CoinGecko universe

In [ ]:
matched = tokens_df[tokens_df['symbol'].str.upper().isin(cg_symbols)]
print(f'intersection: {len(matched)} tokens covered by both CoinGecko and tokenomist')
matched[['id', 'symbol', 'name', 'circulatingSupply', 'totalLockedAmount', 'unlockedAmount']].head(15)

## 4. Historical unlock events for the intersection

In [ ]:
events_df = tokenomist.fetch_all_unlock_events(matched['id'].tolist())
events_df.to_csv('../data/tokenomist_unlock_events.csv', index=False)
print(f'fetched {len(events_df)} (event, allocation) rows across {events_df["tokenSymbol"].nunique()} tokens')
events_df.head(10)

## 5. Supplementary figures

In [ ]:
plots.fig3_unlock_calendar(events_df, '../figures/fig3_unlock_calendar.png')
Image('../figures/fig3_unlock_calendar.png')

In [ ]:
matched_supply = supply_df[supply_df['symbol'].str.upper().isin(cg_symbols)]
plots.fig4_locked_share(matched_supply, '../figures/fig4_locked_share.png')
Image('../figures/fig4_locked_share.png')

## 6. Cumulative dilution over the lookback window

For each token, sum tokenomist's per-event `valueToMarketCap` across all unlock events in the past 365 days. This is a units-friendly approximation of *how much supply, valued in market-cap-equivalent terms, has flooded the float over the period*. Values close to or above 100% indicate that cumulative released supply over the period was worth roughly the same as the token's market capitalization — i.e., the float roughly doubled in market-cap terms.

Caveat: the denominator (mcap) varies across events, so this is a magnitude rather than a strict ratio. Read it as an order-of-magnitude indicator of supply pressure, not as a precise dilution percentage.

In [ ]:
dilution = tokenomist.cumulative_dilution(events_df, lookback_days=365)
dilution.to_csv('../data/tokenomist_cumulative_dilution.csv', index=False)
dilution

In [ ]:
plots.fig5_cumulative_dilution(dilution, '../figures/fig5_cumulative_dilution.png')
Image('../figures/fig5_cumulative_dilution.png')

## What this notebook shows

- **Locked-share spread (fig4):** the supply-side overhang varies widely across covered tokens, with several at >40% of trackable supply still locked. This cross-sectional spread is what makes an unlock event-study identifiable in principle.
- **Cumulative dilution (fig5):** in the four-token intersection sample, three of four tokens absorbed cumulative supply releases worth more than 40% of their market cap over the past year, with OP exceeding 100%. This is the supply-side parallel to the price-side mismeasurement: even setting aside questions of liquidity, reported market cap can mask large recurring float expansions.
- **Unlock-event coverage (fig3):** the pipeline can pull, parse, and align scheduled unlock events with the CoinGecko price/volume series at the symbol level. The full project will scale this up using paid-tier API access and run the actual event-study regressions on realized post-event returns.